<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/Prompt Runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [1]:
%pip install neo4j -q
%pip install -q -U torchinfo
%pip install -q -U bitsandbytes
%pip install -q -U accelerate
%pip install -U bitsandbytes

In [2]:
import pandas as pd
from datetime import datetime, time
import re

from neo4j import GraphDatabase
import math, os

from typing import List, Dict, Any
import pandas as pd
import textwrap
import json
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

try:
    from google.colab import userdata
    get_secret = userdata.get
except ImportError:
    # Fallback: define a dummy get_secret for local use
    def get_secret(key):
        return None

from dotenv import load_dotenv
load_dotenv()


from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

def get_env(key: str, default: str = None):
    """Tries os.getenv first, then Colab userdata if available."""
    return os.getenv(key) or get_secret(key) or default

NEO4J_URI  = get_env("NEO4J_URI")   # bolt:// or neo4j://
NEO4J_USER = get_env("NEO4J_USER")
NEO4J_PASS = get_env("NEO4J_PASS")
NEO4J_DATABASE = get_env("NEO4J_DATABASE")

DB_HOST = get_env("POSTGRES_DB_HOST")
DB_PORT = int(get_env("POSTGRES_DB_PORT"))
DB_NAME = get_env("POSTGRES_DB_NAME")
DB_USER = get_env("POSTGRES_DB_USER")
DB_PASS = get_env("POSTGRES_DB_PASS")

connection_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

from typing import Optional, Iterable, Dict, Any, List
from neo4j import GraphDatabase
import pandas as pd
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

In [3]:
import time
from typing import List, Optional

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM


def load_hf_model(model_path: str):
    """
    Load tokenizer + model once and reuse across calls to run_hf_llm.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    model.eval()

    # Ensure pad tokens are set
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    # Small speed boost on modern GPUs
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True

    return tokenizer, model


In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


def load_hf_model_with_lora(
    base_model_id: str,
    lora_path: str,
):
    """
    Load Gemma base model + LoRA adapter and return (tokenizer, model)
    compatible with run_hf_llm.
    """

    tokenizer = AutoTokenizer.from_pretrained(
        base_model_id, use_fast=True, fix_mistral_regex=True)
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "right"



    # 4-bit quantized base model (same style as QLoRA training)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )

    # Attach LoRA adapter
    model = PeftModel.from_pretrained(base_model, lora_path)
    model.eval()

    # Make sure pad tokens are set
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True

    return tokenizer, model


In [5]:
def run_hf_llm(
    df: pd.DataFrame,
    tokenizer,
    model,
    prompt_col: str = "prompt_text",
    system_prompt: str = "",
    system_prompt_col: Optional[str] = None,
    max_new_tokens: int = 256,
    temperature: float = 0.2,
    top_p: float = 0.9,
    messages: Optional[List[dict]] = None,
    chat_format: Optional[str] = None,
    strip_think_tags: bool = True,
    batch_size: int = 4,
) -> pd.DataFrame:
    device = model.device
    base_messages = messages[:] if messages else []

    def render_with_chat_template(msgs: List[dict]) -> str:
        if hasattr(tokenizer, "apply_chat_template"):
            try:
                return tokenizer.apply_chat_template(
                    msgs,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            except Exception as e:
                err = str(e)
                if "System role not supported" in err or "system" in err.lower():
                    system_text = "\n\n".join(
                        m["content"] for m in msgs if m.get("role") == "system"
                    ).strip()
                    fixed = []
                    merged = False
                    for m in msgs:
                        role = m.get("role")
                        content = m.get("content", "")
                        if role == "system":
                            continue
                        if role == "user" and system_text and not merged:
                            fixed.append(
                                {
                                    "role": "user",
                                    "content": system_text + "\n\n" + content,
                                }
                            )
                            merged = True
                        else:
                            fixed.append(m)

                    return tokenizer.apply_chat_template(
                        fixed,
                        tokenize=False,
                        add_generation_prompt=True,
                    )

        parts = []
        if system_prompt:
            parts.append(system_prompt.strip())
        for m in base_messages:
            role = m.get("role", "user").upper()
            parts.append(f"{role}:\n{m.get('content', '').strip()}")
        return "\n\n".join(parts)

    def build_prompt_text(user_prompt: str, system_prompt_inp: str) -> str:
        row_messages: List[dict] = []

        if system_prompt_inp and not any(
            m.get("role") == "system" for m in base_messages
        ):
            row_messages.append({"role": "system", "content": system_prompt_inp})

        row_messages.extend(base_messages)
        row_messages.append({"role": "user", "content": user_prompt})

        rendered = render_with_chat_template(row_messages)

        if "ASSISTANT:" in rendered:
            return rendered

        return rendered + f"\n\nUSER:\n{user_prompt.strip()}\nASSISTANT:\n"

    # ---- 1) Build all prompts once ----
    prompts: List[str] = []
    for _, row in df.iterrows():
        user_prompt = str(row[prompt_col])
        system_prompt_inp = (
            str(row[system_prompt_col]) if system_prompt_col else system_prompt
        )
        prompts.append(build_prompt_text(user_prompt, system_prompt_inp))

    outputs: List[str] = []
    time_taken: List[float] = []
    token_counts: List[int] = []

    model.eval()

    # ---- max_length for truncation (fix for OverflowError) ----
    model_max_len = getattr(tokenizer, "model_max_length", None)
    if (
        model_max_len is None
        or model_max_len == float("inf")
        or model_max_len > 1_000_000
    ):
        model_max_len = 8192
    model_max_len = int(model_max_len)

    # ---- 2) Generate in batches ----
    from tqdm.auto import tqdm

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch_prompts = prompts[i : i + batch_size]

        enc = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=model_max_len,
        ).to(device)

        start = time.time()
        with torch.inference_mode():
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=model.config.pad_token_id,
            )
        end = time.time()

        input_len = enc["input_ids"].shape[1]
        gen_only_ids = gen_ids[:, input_len:]

        batch_texts = tokenizer.batch_decode(
            gen_only_ids, skip_special_tokens=True
        )

        per_sample_time = round((end - start) / len(batch_prompts), 2)

        for txt, gen_ids_row in zip(batch_texts, gen_only_ids):
            if strip_think_tags and txt:
                txt = (
                    txt.replace("<think>", "")
                    .replace("</think>", "")
                    .strip()
                )

            outputs.append(txt)
            token_counts.append(int((gen_ids_row != tokenizer.pad_token_id).sum().item()))
            time_taken.append(per_sample_time)

    out_df = df.copy()
    out_df["output"] = outputs
    out_df["time_taken"] = time_taken
    out_df["token_count"] = token_counts
    return out_df


In [6]:
prompt_df = pd.read_sql("SELECT * FROM policy_prompts", engine)

In [7]:
model_path = "google/gemma-2-9b-it"  # or your local path

# tokenizer, model = load_hf_model(model_path)

# result_df = run_hf_llm(
#     df=prompt_df,
#     tokenizer=tokenizer,
#     model=model,
#     prompt_col="user_prompt",
#     system_prompt=None,
#     system_prompt_col="system_prompt",    # or a column name if you have per-row system prompts
#     max_new_tokens=256,
#     temperature=0.2,
#     top_p=0.9,
#     messages=None,             # or your few-shot messages list
#     batch_size=16,              # tweak depending on VRAM
# )



In [8]:
# result_df.to_sql("prompt_responses", engine, if_exists="replace", index=False)

In [9]:
test_df = pd.read_sql("""SELECT prompt_responses.*, policy_splits.split_label, policy.org_text FROM prompt_responses
JOIN policy_splits on policy_splits.policy_id = prompt_responses.policy_id
JOIn(
SELECT DISTINCT policy_id, section_id, section_title, source_framework, string_agg(clause_text, '\n' ORDER BY ROW_NUMBER) AS org_text  FROM policy_lines
GROUP BY policy_id, section_id, section_title, source_framework
) as policy on prompt_responses.section_id = policy.section_id
WHERE version = 'v2.5' AND split_label = 'test'

""", engine)

In [10]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [15]:
base_model_id = "google/gemma-2-9b-it"
lora_path = "/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model"


tokenizer, model = load_hf_model_with_lora(
    base_model_id,
    lora_path
)

dpo_result_df = run_hf_llm(
    df=test_df,
    tokenizer=tokenizer,
    model=model,
    prompt_col="user_prompt",
    system_prompt=None,
    system_prompt_col="system_prompt",    # or a column name if you have per-row system prompts
    max_new_tokens=256,
    temperature=0.2,
    top_p=0.9,
    messages=None,             # or your few-shot messages list
    batch_size=64,              # tweak depending on VRAM
)

dpo_result_df['version'] = 'v4.1'

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/266/gemma2_9b_it_v2_5_lora_dpo_2/gemma_merged_full_model'

In [ ]:
dpo_result_df.to_sql("prompt_responses2", engine, if_exists="append", index=False)